<a href="https://colab.research.google.com/github/peremartra/fairness-pruning/blob/main/notebooks/02_Evaluate_BBQ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fairness Pruning Research - Base Model Evaluation
## 02 - BBQ (English) Baseline for Unpruned Models

### Establishing Bias Performance using BBQ for Bias Mitigation Research
by [Pere Martra](https://github.com/peremartra)

[![GitHub](https://img.shields.io/badge/⭐_Star-OptiPFair-orange?logo=github&logoColor=white)](https://github.com/peremartra/optipfair)
[![PyPI](https://img.shields.io/pypi/v/optipfair?logo=python&logoColor=white&label=v)](https://pypi.org/project/optipfair/)

**Repository:** [github.com/peremartra/fairness-pruning](https://github.com/peremartra/fairness-pruning)

---

**Colab Environment:** GPU A100 recommended (40 GB or more). Full BBQ takes about 56 min for a 7-8B model on an A100-40GB (701,904 requests at about 207 requests/s); an L4 is several times slower.

**Models to Evaluate:**
* Llama-3.2-1B (base)
* Llama-3.2-3B (base)
* Salamandra-2B (base)
* Llama-3.1-8B (base)
* Salamandra-7B (base)
---

**Note:** This notebook evaluates ONLY base models (no pruning applied). For bias mitigation experiments with pruned models, see subsequent notebooks.

---

# 02 — BBQ baseline evaluation (unpruned models)

## Prompt formats

The notebook has a switch, `PROMPT_FORMAT`, in the configuration cell. It selects the task, the checkpoint directory and the prefix of the output files, so results of the two formats are never mixed:

| `PROMPT_FORMAT` | Task | Checkpoints (Drive) | Output files |
|---|---|---|---|
| `standard` | `bbq` (lm-eval default; scores the answer texts, no options shown) | `checkpoints_bbq` | `base_models_bbq_results_*` |
| `abc` | `bbq_base_mc` (custom; options A/B/C + `Answer:`, plain text, no chat template; see `custom_tasks/bbq_instruct/README.md`) | `checkpoints_bbq_abc` | `base_models_bbq_abc_results_*` |
| `standard_instruct` | `bbq` (same task and prompt as `standard`, run on instruct models, plain text, no chat template) | `checkpoints_bbq_standard_instruct` | `instruct_models_bbq_standard_results_*` |
| `standard_rerun` | `bbq` (same task and prompt as `standard`, base model repeated under the current configuration; E2, step S1) | `checkpoints_bbq_rerun` | `base_models_bbq_rerun_results_*` |
| `standard_rerun` | `bbq` (same task and prompt as `standard`, base model repeated under the current configuration; E2, step S1) | `checkpoints_bbq_rerun2` | `base_models_bbq_rerun_results_*` |

Runs with `abc` are for comparison with the instruct notebook (`02_Evaluate_BBQ_Instruct.ipynb`) and are not comparable with `standard` runs. Runs with `standard_instruct` are directly comparable with `standard` runs (same task and prompt, only the model changes): they isolate the effect of instruction tuning from the effect of the A/B/C format. They are run without the chat template (`chat_template` is `False` in `FORMAT_SETTINGS`).

## Evaluation log

| Run | Date | Models | Task | Settings | GPU |
|---|---|---|---|---|---|
| 1 | 2025-12-07 | BSC-LT/salamandra-2b, meta-llama/Llama-3.2-1B, meta-llama/Llama-3.2-3B | `bbq` (English, full dataset) | 0-shot, bfloat16, `batch_size=8`, lm-eval version not recorded | not recorded |
| 2 | 2026-10-27 | meta-llama/Llama-3.1-8B | `bbq` (English, full dataset) | 0-shot, bfloat16, `batch_size=16`, lm-eval 0.4.9.2, torch 2.10.0, transformers 4.57.3 | NVIDIA A100 (memory size not recorded) |
| 3 | 2026-10-28 | BSC-LT/salamandra-7b | `bbq` (English, full dataset) | same as run 2; 701,904 requests, 56 min | NVIDIA A100-SXM4-40GB |
| 4 | 2026-10-29 | Qwen/Qwen3-4B-Base | `bbq` (English, full dataset) | same as run 2; 701,904 requests, 47 min | NVIDIA A100-SXM4-40GB |
| 5 | 2026-10-30 | meta-llama/Llama-3.2-3B | `bbq_base_mc` (A/B/C, `PROMPT_FORMAT = "abc"`, English, full dataset) | same as run 2; 175,476 requests, 4 min 25 s | NVIDIA A100-SXM4-40GB |
| 6 | 2026-10-30 | BSC-LT/salamandra-7b | `bbq_base_mc` (A/B/C) | same as run 5; duration and GPU not recorded (the notebook was not saved with this run's output) | not recorded |
| 7 | 2026-10-30 | meta-llama/Llama-3.1-8B | `bbq_base_mc` (A/B/C) | same as run 5; 175,476 requests, 8 min 23 s | NVIDIA A100-SXM4-40GB |
| 8 | 2026-10-30 | meta-llama/Llama-3.2-3B-Instruct, meta-llama/Llama-3.1-8B-Instruct | `bbq` (`PROMPT_FORMAT = "standard_instruct"`, no chat template) | same as run 2; duration and GPU not recorded | not recorded |
| 9 | 2026-11-02 | meta-llama/Llama-3.2-3B | `bbq` (`PROMPT_FORMAT = "standard_rerun"`, English, full dataset; repetition of run 1 under the current configuration) | same as run 2 (lm-eval 0.4.9.2, torch 2.10.0, transformers 4.57.3); 39 min 11 s (10:08:57 to 10:48:08), number of requests not recorded | NVIDIA A100-SXM4-40GB |

Notes:
- Run 1 was executed with the notebook version of 2025-12-08 (commit "BBQ results obtained"), before `requirements.txt` existed (created 2026-02-23). Since then `utils.py` changed the default batch size to `"auto"` (2026-05-15) and `requirements.txt` pins lm-eval 0.4.9.2. Runs 2 and 3 use the current configuration, with `batch_size=16` fixed in the run cell. Dates are those of the files saved on Drive (UTC).
- Run 1 baseline results (overall accuracy / ambiguous / disambiguated):
  - Salamandra-2B: 29.05 % / 7.66 % / 50.45 %
  - Llama-3.2-1B: 31.15 % / 10.01 % / 52.28 %
  - Llama-3.2-3B: 40.52 % / 7.51 % / 73.54 %
- Runs 2 and 3 baseline results (overall accuracy / ambiguous / disambiguated):
  - Llama-3.1-8B: 45.19 % / 8.35 % / 82.02 %
  - Salamandra-7B: 44.02 % / 7.65 % / 80.40 %
- Aggregates are those reported by lm-eval for the whole `bbq` task (pooled over all 58,492 items; ambiguous and disambiguated items are 50 % each).
- Run 1 and runs 2-3 differ in batch size (8 vs 16), library versions and GPU. The Llama-3.2-3B baseline was repeated under the current configuration (run 9, below) to quantify that effect.
- Run 9 (Llama-3.2-3B, `bbq`, `PROMPT_FORMAT = "standard_rerun"`, same task and prompt as run 1, current configuration; 2026-11-02, NVIDIA A100-SXM4-40GB, lm-eval 0.4.9.2, torch 2.10.0, transformers 4.57.3, 39 min 11 s; overall accuracy / ambiguous / disambiguated; ambiguous / disambiguated bias score):
  - Run 9: 40.51 % / 7.48 % / 73.53 %; 4.03 % / 2.37 % (run 1: 40.52 / 7.51 / 73.54; 4.12 / 2.24)
  - By category (bias score, run 1 → run 9, ambiguous / disambiguated, in %): Age 16.30 → 15.65 / 8.48 → 8.91; Disability status 4.63 → 4.11 / 0.51 → 0.51; Gender identity 10.86 → 11.00 / 8.04 → 8.25; Nationality 4.74 → 4.61 / 3.52 → 3.79; Physical appearance 17.89 → 17.77 / 9.02 → 8.67; Race/ethnicity 1.34 → 0.76 / 1.34 → 0.99; Race x gender 1.40 → 1.30 / 0.50 → 0.55; Race x SES 1.52 → 1.77 / -0.69 → 0.06; Religion 6.00 → 6.00 / 3.84 → 3.84; SES 1.37 → 1.57 / 0.17 → 0.29; Sexual orientation 4.86 → 4.17 / 0.93 → 0.93.
  - The aggregates differ from run 1 by at most 0.03 pp (accuracy) and 0.13 pp (bias score); by category the largest difference is 0.75 pp (Race x SES, disambiguated) with mixed signs, and Religion is identical. The cause is not attributed: batch size (8 → 16), library versions and possibly the GPU changed. Per-category standard errors are N/A in the lm-eval aggregates. Run 9 is the noise reference for the zeroing experiments of E2 (see `R1_exp.md`).
  - Files: checkpoint `fair_pruning/checkpoints_bbq_rerun/3b/meta_llama_llama_3.2_3b.json`; per-item predictions `fair_pruning/checkpoints_bbq_rerun/results/lm_evals/meta_llama_llama_3.2_3b_bbq.json` (about 700 MB; opened on 2026-11-02: 58,492 items with 12 scored options each; accuracy and bias per category recomputed from the items match the lm-eval aggregates, 0 item-level mismatches); consolidated files `base_models_bbq_rerun_results_*` in `fair_pruning/results/`. Number of requests not recorded.
- Per-item predictions (`log_samples=True`) of runs 2 and 3 are in the raw lm-eval JSON on Drive: `fair_pruning/checkpoints_bbq/results/lm_evals/<model>_bbq.json` (about 700 MB each, not in this repository). Run 1 has only the aggregate checkpoints (`fair_pruning/checkpoints_bbq/<size>/<model>.json`), no per-item file.
- Run 4 baseline (ambiguous / disambiguated accuracy): Qwen3-4B-Base 15.0 % / 86.7 %.
- Runs 5-7 (A/B/C format, task `bbq_base_mc`; overall accuracy / ambiguous / disambiguated; ambiguous / disambiguated bias score):
  - Llama-3.2-3B: 39.65 % / 7.25 % / 72.06 %; 11.49 % / 4.88 %
  - Salamandra-7B: 28.23 % / 3.74 % / 52.73 %; 2.12 % / 2.63 %. Disambiguated accuracy close to random choice between the two named people and overall accuracy below the 33 % of choosing among three options at random: the base model does not use the A/B/C format as intended, and its bias scores in this format are not interpretable.
  - Llama-3.1-8B: 60.51 % / 38.28 % / 82.74 %; 10.03 % / 5.01 %
- Runs 5-7 are for comparison with the instruct models of `02_Evaluate_BBQ_Instruct.ipynb` (same options A/B/C; they differ in the chat template and in the final `Answer:` line). They are not comparable with the `bbq` runs: the format changes the bias scores (Llama-3.2-3B: ambiguous bias 4.12 % with `bbq` and 11.49 % with A/B/C) and, depending on the model, the ambiguous accuracy (Llama-3.1-8B: 8.35 % with `bbq` and 38.28 % with A/B/C; Llama-3.2-3B: 7.51 % and 7.25 %).
- Run 8 (instruct models, standard `bbq` task and prompt, no chat template; overall accuracy / ambiguous / disambiguated; ambiguous / disambiguated bias score):
  - Llama-3.2-3B-Instruct: 47.21 % / 10.92 % / 83.50 %; 8.77 % / 3.24 % (base, same task: 40.52 / 7.51 / 73.54; 4.12 / 2.24)
  - Llama-3.1-8B-Instruct: 49.41 % / 8.47 % / 90.35 %; 13.87 % / 1.88 % (base, same task: 45.19 / 8.35 / 82.02; 5.59 / 2.08)
  - With the same task and prompt, instruction tuning raises the disambiguated accuracy (+10.0 and +8.3 pp) but not the ambiguous accuracy (+3.4 and +0.1 pp); the ambiguous bias score is higher than in the base models. The large ambiguous-accuracy gain of the instruct models seen in `02_Evaluate_BBQ_Instruct.ipynb` (A/B/C + chat template) is therefore not explained by instruction tuning alone. Not separated: chat template vs letter format (instruct model in A/B/C without chat template was not run). Files: `instruct_models_bbq_standard_results_*` in `fair_pruning/results/` and in `results/`.
- A/B/C checkpoints: `fair_pruning/checkpoints_bbq_abc/<size>/<model>.json`; per-item predictions: `fair_pruning/checkpoints_bbq_abc/results/lm_evals/<model>_bbq_base_mc.json`; consolidated files: `base_models_bbq_abc_results_*.csv/json` in `fair_pruning/results/` and in this repository (`results/`).
- Spanish baselines (EsBBQ, 10 categories) are in `02_Evaluate_MBBQ.ipynb`, not here.


# 1. Setup & Installation

In [1]:
# Clone the repository (light: no history, only notebooks and custom tasks)
import os, subprocess, sys

REPO_URL = "https://github.com/peremartra/fairness-pruning.git"
REPO_DIR = "fairness-pruning"

if not os.path.exists(REPO_DIR):
    !git clone -q --depth 1 --filter=blob:none --sparse {REPO_URL}
    !cd {REPO_DIR} && git sparse-checkout set notebooks custom_tasks
else:
    print("Repo already cloned. Pulling latest changes...")
    !cd {REPO_DIR} && git pull -q

%cd {REPO_DIR}
print(f"Working directory: {os.getcwd()}")

# Install dependencies from requirements.txt
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-r", "requirements.txt"]
)
print("Dependencies installed.")

# Versions actually in use (for the evaluation log)
from importlib.metadata import version
for pkg in ("torch", "transformers", "lm-eval", "accelerate", "datasets"):
    print(f"{pkg:14s} {version(pkg)}")
print("Python:", sys.version.split()[0])
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 9 (delta 0), reused 3 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (9/9), 53.59 KiB | 2.23 MiB/s, done.
remote: Enumerating objects: 72, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 72 (delta 45), reused 47 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (72/72), 6.51 MiB | 23.08 MiB/s, done.
Resolving deltas: 100% (45/45), done.
/content/fairness-pruning
Working directory: /content/fairness-pruning
Dependencies installed.
torch          2.10.0
transformers   4.57.3
lm-eval        0.4.9.2
accelerate     1.13.0
datasets       4.0.0
Python: 3.13.16
NVIDIA A100-SXM4-40GB, 40960 MiB


In [2]:
# Mount Google Drive for checkpoint persistence
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 2. Configuration & Evaluation Plan (BBQ-only)

Configure paths, select the BBQ task, and list the models we will evaluate.

In [3]:
import torch
import json
import pandas as pd
from datetime import datetime
from pathlib import Path
import logging
import os
from transformers import AutoModelForCausalLM, AutoTokenizer
from IPython.display import display
from utils import (
    #EXPERIMENT_CONFIG,
    run_robust_evaluation,
    load_or_create_model,
    clear_gpu_cache,
    format_results_table,
    get_model_stats,
 )

# Prompt format:
#   "standard"          -> lm-eval's default `bbq` task (scores the answer texts, no options shown), base models
#   "abc"               -> custom task `bbq_base_mc` (options A/B/C + "Answer:", plain text, no chat template), base models
#   "standard_instruct" -> the same default `bbq` task and prompt as "standard", run on instruct models
#                          (plain text, no chat template); separate directory and file prefix so that
#                          instruct models do not mix with the base-model results
#   "standard_rerun"   -> the same default `bbq` task and prompt as "standard", repeated on a base model under the current
#                          configuration (E2, step S1); separate directory so that completed checkpoints of
#                          earlier runs are not skipped and the original results are not overwritten
PROMPT_FORMAT = "standard_rerun"   # "standard" | "abc" | "standard_instruct" | "standard_rerun"

FORMAT_SETTINGS = {
    "standard": {
        "task": "bbq",
        "checkpoint_dir": "/content/drive/MyDrive/fair_pruning/checkpoints_bbq",
        "output_prefix": "base_models_bbq",
        "chat_template": False,
    },
    "abc": {
        "task": "bbq_base_mc",
        "checkpoint_dir": "/content/drive/MyDrive/fair_pruning/checkpoints_bbq_abc",
        "output_prefix": "base_models_bbq_abc",
        "chat_template": False,
    },
    "standard_instruct": {
        "task": "bbq",
        "checkpoint_dir": "/content/drive/MyDrive/fair_pruning/checkpoints_bbq_standard_instruct",
        "output_prefix": "instruct_models_bbq_standard",
        "chat_template": False,
    },
    "standard_rerun": {
        "task": "bbq",
        "checkpoint_dir": "/content/drive/MyDrive/fair_pruning/checkpoints_bbq_rerun2",
        "output_prefix": "base_models_bbq_rerun",
        "chat_template": False,
    },
}
assert PROMPT_FORMAT in FORMAT_SETTINGS, f"Unknown PROMPT_FORMAT: {PROMPT_FORMAT}"
_fmt = FORMAT_SETTINGS[PROMPT_FORMAT]

# Paths (Drive recommended in Colab)
CHECKPOINT_BASE_DIR = _fmt["checkpoint_dir"]
OUTPUT_PREFIX = _fmt["output_prefix"]
APPLY_CHAT_TEMPLATE = _fmt["chat_template"]
RESULTS_DIR = "/content/drive/MyDrive/fair_pruning/results"
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

# BBQ task list (0-shot)
BBQ_TASKS = [{"name": _fmt["task"], "num_fewshot": 0}]

# De-duplicate models from EXPERIMENT_CONFIG
EXPERIMENT_CONFIG = [
    {
        "base_model": "meta-llama/Llama-3.2-3B",
    },
]
unique_models = list(dict.fromkeys([cfg["base_model"] for cfg in EXPERIMENT_CONFIG]))

logging.getLogger("lm_eval").setLevel(logging.INFO)

print(f"{'='*70}")
print(f"📊 EVALUATION PLAN: BBQ-only (prompt format: {PROMPT_FORMAT})")
print(f"{'='*70}")
print(f"Checkpoints: {CHECKPOINT_BASE_DIR}")
print(f"Results: {RESULTS_DIR} (file prefix: {OUTPUT_PREFIX})")
print(f"Tasks: {[task['name'] for task in BBQ_TASKS]} | chat template: {APPLY_CHAT_TEMPLATE}")
print("Models:")
for m in unique_models:
    print(f" - {m}")
print(f"{'='*70}\n")

📊 EVALUATION PLAN: BBQ-only (prompt format: standard_rerun)
Checkpoints: /content/drive/MyDrive/fair_pruning/checkpoints_bbq_rerun2
Results: /content/drive/MyDrive/fair_pruning/results (file prefix: base_models_bbq_rerun)
Tasks: ['bbq'] | chat template: False
Models:
 - meta-llama/Llama-3.2-3B



In [4]:
# Register the custom task bbq_base_mc in lm-eval (only needed for PROMPT_FORMAT = "abc")
import shutil
import lm_eval

if PROMPT_FORMAT == "abc":
    TASK_SRC = "custom_tasks/bbq_instruct"
    TASK_DST = os.path.join(os.path.dirname(lm_eval.__file__), "tasks", "bbq_instruct")

    shutil.rmtree(TASK_DST, ignore_errors=True)
    shutil.copytree(TASK_SRC, TASK_DST)
    print(f"Task installed in: {TASK_DST}")
    print("Files:", sorted(os.listdir(TASK_DST)))
else:
    print(f"PROMPT_FORMAT = {PROMPT_FORMAT}: using lm-eval's default `bbq` task, nothing to register.")

PROMPT_FORMAT = standard_rerun: using lm-eval's default `bbq` task, nothing to register.


# 3. Helper Functions

Utility functions for automatic checkpoint path generation and model size detection.

In [5]:
import re
import os

def get_model_size(model_name: str) -> str:
    """Extract model size identifier from HuggingFace model name.

    Examples:
        "meta-llama/Llama-3.2-1B" → "1b"
        "meta-llama/Llama-3.2-3B-Instruct" → "3b_instruct"
        "BSC-LT/salamandra-2b" → "2b"
    """
    match = re.search(r'(\d+\.?\d*)[Bb]', model_name)
    if not match:
        return "unknown"

    size = match.group(1).replace('.', '_') + "b"
    if "instruct" in model_name.lower():
        size += "_instruct"

    return size.lower()

def get_checkpoint_path(model_name: str, base_dir: str) -> str:
    """Generate checkpoint path with size-based subdirectory.

    Args:
        model_name: Full HuggingFace model identifier
        base_dir: Base directory for checkpoints

    Returns:
        Full path to checkpoint file
    """
    model_size = get_model_size(model_name)
    safe_name = model_name.replace('/', '_').replace('-', '_').lower()
    checkpoint_dir = os.path.join(base_dir, model_size)
    os.makedirs(checkpoint_dir, exist_ok=True)
    return os.path.join(checkpoint_dir, f"{safe_name}.json")

# Test with EXPERIMENT_CONFIG
print("Testing helper functions with EXPERIMENT_CONFIG:")
print("-" * 70)
for cfg in EXPERIMENT_CONFIG:
    model_id = cfg['base_model']
    size = get_model_size(model_id)
    print(f"{model_id:<50} → {size}")
print("-" * 70)

Testing helper functions with EXPERIMENT_CONFIG:
----------------------------------------------------------------------
meta-llama/Llama-3.2-3B                            → 3b
----------------------------------------------------------------------


# 4. Run BBQ Evaluation

Evaluate each base model on the BBQ task with checkpoint/resume and raw result saving.

In [6]:
print(f"\n{'='*70}")
print("🚀 STARTING BBQ EVALUATION (Hugging Face Native Loading)")
print(f"{'='*70}\n")

all_model_results = {}

for idx, model_id in enumerate(unique_models, 1):
    print(f"\n{'='*70}")
    print(f"📊 MODEL {idx}/{len(unique_models)}: {model_id}")
    print(f"{'='*70}\n")
    try:
        # Generar ruta de checkpoint
        checkpoint_path = get_checkpoint_path(model_id, CHECKPOINT_BASE_DIR)

        # --- CARGA DIRECTA CON HUGGING FACE ---
        print(f"📥 Loading directly from Hugging Face Hub...")
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.bfloat16,
            device_map="auto"
        )

        tokenizer = AutoTokenizer.from_pretrained(model_id)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        print("✅ Model loaded successfully (Native HF)")

        # Mostrar estadísticas
        stats = get_model_stats(model)
        print(f"📈 Params: {stats['total_parameters']:,} | Size: {stats['size_gb']:.2f} GB")
        print(f"📁 Checkpoint: {checkpoint_path}\n")

        # --- Run evaluation: checkpoint saved after each task (resume on re-run);
        # raw per-item lm-eval JSON per task in <CHECKPOINT_BASE_DIR>/results/lm_evals/ ---
        results = run_robust_evaluation(
            model=model,
            tokenizer=tokenizer,
            tasks=BBQ_TASKS,
            checkpoint_path=checkpoint_path,
            model_name=model_id,
            batch_size=16,
            apply_chat_template=APPLY_CHAT_TEMPLATE
        )
        # -----------------------------------------------

        all_model_results[model_id] = results
        print(f"\n✅ Completed: {model_id}")
        print("Results Preview:")
        print(format_results_table(results))

    except Exception as e:
        print(f"\n❌ ERROR evaluating {model_id}: {e}")
        import traceback
        traceback.print_exc()
        if 'model' in locals(): del model
        if 'tokenizer' in locals(): del tokenizer
        clear_gpu_cache()
        continue

    # Limpieza de memoria
    del model
    del tokenizer
    clear_gpu_cache()

print(f"\n{'='*70}")
print(f"✅ BBQ EVALUATION COMPLETE: {len(all_model_results)}/{len(unique_models)} models")
print(f"{'='*70}\n")


🚀 STARTING BBQ EVALUATION (Hugging Face Native Loading)


📊 MODEL 1/1: meta-llama/Llama-3.2-3B

📥 Loading directly from Hugging Face Hub...


config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

... (11 outputs left out)

Generating test split:   0%|          | 0/58492 [00:00<?, ? examples/s]

Map:   0%|          | 0/58492 [00:00<?, ? examples/s]

INFO:lm_eval.tasks:Selected tasks:
INFO:lm_eval.tasks:Task: bbq (bbq/bbq_multiple_choice.yaml)
INFO:lm_eval.api.task:Building contexts for bbq on rank 0...
100%|██████████| 58492/58492 [00:37<00:00, 1540.79it/s]
INFO:lm_eval.evaluator:Running loglikelihood requests
Running loglikelihood requests: 100%|██████████| 701904/701904 [31:57<00:00, 366.02it/s]



💾 Saving raw results to: /content/drive/MyDrive/fair_pruning/checkpoints_bbq_rerun2/results/lm_evals
   ✅ Saved: meta_llama_llama_3.2_3b_bbq.json
✅ bbq completed and saved to checkpoint
   Results: {'alias': 'bbq', 'acc': '0.4051', 'acc_stderr': '0.0020', 'accuracy_amb': '0.0748', 'accuracy_amb_stderr': 'N/A', 'accuracy_disamb': '0.7353', 'accuracy_disamb_stderr': 'N/A', 'amb_bias_score': '0.0403', 'amb_bias_score_stderr': 'N/A', 'disamb_bias_score': '0.0237', 'disamb_bias_score_stderr': 'N/A', 'amb_bias_score_Age': '0.1565', 'amb_bias_score_Age_stderr': 'N/A', 'disamb_bias_score_Age': '0.0891', 'disamb_bias_score_Age_stderr': 'N/A', 'amb_bias_score_Disability_status': '0.0411', 'amb_bias_score_Disability_status_stderr': 'N/A', 'amb_bias_score_Gender_identity': '0.1100', 'amb_bias_score_Gender_identity_stderr': 'N/A', 'amb_bias_score_Nationality': '0.0461', 'amb_bias_score_Nationality_stderr': 'N/A', 'amb_bias_score_Physical_appearance': '0.1777', 'amb_bias_score_Physical_appearance_s

# 5. Consolidate BBQ Results

Load checkpoint files, flatten metrics, and export combined BBQ results.

In [7]:
import glob

def flatten_metrics(metrics, prefix=''):
    flat = {}
    for k, v in metrics.items():
        if isinstance(v, dict):
            flat.update(flatten_metrics(v, prefix=f"{prefix}{k}_"))
        else:
            flat[f"{prefix}{k}"] = v
    return flat

print(f"{'='*70}")
print(" CONSOLIDATING BBQ RESULTS")
print(f"{'='*70}\n")

checkpoint_files = [
    f for f in glob.glob(f"{CHECKPOINT_BASE_DIR}/**/*.json", recursive=True)
    if "lm_evals" not in f
]
print(f"Total checkpoint JSONs: {len(checkpoint_files)}")

consolidated_data = []

for json_path in sorted(checkpoint_files):
    if "lm_evals" in json_path:
        continue  # skip raw per-task dumps
    try:
        with open(json_path, "r") as f:
            data = json.load(f)

        metadata = data.get("metadata", {})
        model_name = metadata.get("model_name", "Unknown")

        model_size = get_model_size(model_name)

        results = data.get("results", {})
        if not results:
            continue

        for task_name, metrics in results.items():
            row = {
                "model": model_name,
                "model_size": model_size,
                "task": task_name,
            }
            row.update(flatten_metrics(metrics))
            if "accuracy" not in row and "acc" in row:
                row["accuracy"] = row["acc"]
            consolidated_data.append(row)

    except Exception as e:
        print(f"   -> Error reading {json_path}: {e}")

if consolidated_data:
    df = pd.DataFrame(consolidated_data)
    df = df.sort_values(by=["model", "task"]).reset_index(drop=True)
    display(df.head())

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_path = f"{RESULTS_DIR}/{OUTPUT_PREFIX}_results_{timestamp}.csv"
    latest_csv = f"{RESULTS_DIR}/{OUTPUT_PREFIX}_results_latest.csv"
    json_path = f"{RESULTS_DIR}/{OUTPUT_PREFIX}_results_{timestamp}.json"

    df.to_csv(csv_path, index=False)
    df.to_csv(latest_csv, index=False)
    df.to_json(json_path, orient="records", indent=2)

    print("\n💾 BBQ results saved:")
    print(f"   {csv_path}")
    print(f"   {latest_csv}")
    print(f"   {json_path}")
else:
    print("⚠️ No BBQ results found to consolidate.")

 CONSOLIDATING BBQ RESULTS

Total checkpoint JSONs: 1


,model,model_size,task,alias,acc,acc_stderr,accuracy_amb,accuracy_amb_stderr,accuracy_disamb,accuracy_disamb_stderr,...,disamb_bias_score_Race_x_gender_stderr,disamb_bias_score_Race_x_SES,disamb_bias_score_Race_x_SES_stderr,disamb_bias_score_Religion,disamb_bias_score_Religion_stderr,disamb_bias_score_SES,disamb_bias_score_SES_stderr,disamb_bias_score_Sexual_orientation,disamb_bias_score_Sexual_orientation_stderr,accuracy
0,meta-llama/Llama-3.2-3B,3b,bbq,bbq,0.4051,0.0020,0.0748,N/A,0.7353,N/A,...,N/A,0.0006,N/A,0.0384,N/A,0.0029,N/A,0.0093,N/A,0.4051



💾 BBQ results saved:
   /content/drive/MyDrive/fair_pruning/results/base_models_bbq_rerun_results_20261103_095228.csv
   /content/drive/MyDrive/fair_pruning/results/base_models_bbq_rerun_results_latest.csv
   /content/drive/MyDrive/fair_pruning/results/base_models_bbq_rerun_results_20261103_095228.json


# 6. Summary Analysis

Quick per-model accuracy summary for BBQ and file inventory.

In [8]:
# Summary stats
summary_df = None
if 'df' in locals() and not df.empty:
    summaries = []
    for model_name, model_df in df.groupby('model'):
        acc_series = pd.to_numeric(model_df.get('accuracy'), errors='coerce').dropna()
        summaries.append({
            "model": model_name,
            "model_size": model_df.get('model_size', pd.Series(['unknown'])).iloc[0],
            "avg_accuracy": acc_series.mean() if len(acc_series) else None,
            "tasks_completed": len(model_df)
        })

    summary_df = pd.DataFrame(summaries)
    summary_df = summary_df.sort_values("model").reset_index(drop=True)
    print(f"{'='*70}")
    print("📈 BBQ SUMMARY")
    print(f"{'='*70}")
    print(summary_df.to_string(index=False, float_format="%.4f"))

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    summary_csv = f"{RESULTS_DIR}/{OUTPUT_PREFIX}_summary_{timestamp}.csv"
    summary_df.to_csv(summary_csv, index=False)
    print(f"\n💾 Summary saved: {summary_csv}\n")
else:
    print("No consolidated data available for summary.")

print(f"{'='*70}")
print("📁 GENERATED FILES")
print(f"{'='*70}")

if 'csv_path' in locals() and os.path.exists(csv_path):
    print(f"  ✅ {csv_path}")
if 'latest_csv' in locals() and os.path.exists(latest_csv):
    print(f"  ✅ {latest_csv}")
if 'json_path' in locals() and os.path.exists(json_path):
    print(f"  ✅ {json_path}")
if 'summary_csv' in locals() and os.path.exists(summary_csv):
    print(f"  ✅ {summary_csv}")

print("\nCheckpoints (first 10):")
if 'checkpoint_files' in locals():
    for f in sorted(checkpoint_files)[:10]:
        print(f"  ✅ {f}")
    if len(checkpoint_files) > 10:
        print(f"  ... and {len(checkpoint_files) - 10} more")

print(f"{'='*70}")

📈 BBQ SUMMARY
                  model model_size  avg_accuracy  tasks_completed
meta-llama/Llama-3.2-3B         3b        0.4051                1

💾 Summary saved: /content/drive/MyDrive/fair_pruning/results/base_models_bbq_rerun_summary_20261103_095230.csv

📁 GENERATED FILES
  ✅ /content/drive/MyDrive/fair_pruning/results/base_models_bbq_rerun_results_20261103_095228.csv
  ✅ /content/drive/MyDrive/fair_pruning/results/base_models_bbq_rerun_results_latest.csv
  ✅ /content/drive/MyDrive/fair_pruning/results/base_models_bbq_rerun_results_20261103_095228.json
  ✅ /content/drive/MyDrive/fair_pruning/results/base_models_bbq_rerun_summary_20261103_095230.csv

Checkpoints (first 10):
  ✅ /content/drive/MyDrive/fair_pruning/checkpoints_bbq_rerun2/3b/meta_llama_llama_3.2_3b.json
